In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
df = pd.read_csv('Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv')
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [28]:
print(df.shape)
print(df['segment'].value_counts())
df.isnull().sum()

(64000, 12)
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

In [29]:
print("\nCat cols:")
for col in ['history_segment', 'zip_code', 'channel']:
    print(f"\n{col}:\n", df[col].value_counts())


Cat cols:

history_segment:
 history_segment
1) $0 - $100        22970
2) $100 - $200      14254
3) $200 - $350      12289
4) $350 - $500       6409
5) $500 - $750       4911
6) $750 - $1,000     1859
7) $1,000 +          1308
Name: count, dtype: int64

zip_code:
 zip_code
Surburban    28776
Urban        25661
Rural         9563
Name: count, dtype: int64

channel:
 channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64


In [30]:
df.describe()

,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


In [31]:
summary = df.groupby('segment')[['visit','conversion','spend']].mean().round(4)
print(summary)

                visit  conversion   spend
segment                                  
Mens E-Mail    0.1828      0.0125  1.4226
No E-Mail      0.1062      0.0057  0.6528
Womens E-Mail  0.1514      0.0088  1.0772


# 1. A/B Testing

In [32]:
from scipy.stats import chi2_contingency

ct = pd.crosstab(df['segment'], df['conversion'])
print(ct)

conversion         0    1
segment                  
Mens E-Mail    21040  267
No E-Mail      21184  122
Womens E-Mail  21198  189


In [33]:
chi2, p, dof, expected = chi2_contingency(ct)

print(f"Chi2 stats: {chi2:.4f}")
print(f"P-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

Chi2 stats: 55.2580
P-value: 0.000000
Degrees of freedom: 2


In [34]:
from scipy.stats import chi2_contingency

pairs = [("Mens E-Mail", "No E-Mail"),
        ("Womens E-Mail", "No E-Mail"),
        ("Mens E-Mail", "Womens E-Mail")]

for g1, g2 in pairs:
    subset = ct.loc[[g1, g2]]
    chi2, p, _, _ = chi2_contingency(subset)
    print(f"{g1} vs {g2} → p-value: {p:.6f}")   

Mens E-Mail vs No E-Mail → p-value: 0.000000
Womens E-Mail vs No E-Mail → p-value: 0.000197
Mens E-Mail vs Womens E-Mail → p-value: 0.000247


# 2. RFM Segmentation

In [35]:
df['R_score'] = pd.qcut(df['recency'], q=4, labels=[4,3,2,1])
df['M_score'] = pd.qcut(df['history'].rank(method='first'), q=4, labels=[1,2,3,4])

df[['recency','R_score','history','M_score']].head(10)

,recency,R_score,history,M_score
0,10,1,142.44,2
1,6,3,329.08,4
2,7,2,180.65,3
3,9,2,675.83,4
4,2,4,45.34,1
5,6,3,134.83,2
6,9,2,280.20,3
7,9,2,46.42,1
8,9,2,675.07,4
9,10,1,32.84,1


In [36]:
df['R_score'] = df['R_score'].astype(int)
df['M_score'] = df['M_score'].astype(int)

def assign_segment(row):
    r = row['R_score']
    m = row['M_score']
    if r >= 3 and m >= 3:
        return 'Champions'
    elif r >= 3 and m < 3:
        return 'Loyal'
    elif r < 3 and m >= 3:
        return 'At Risk'
    else:
        return 'Lost'

df['rfm_segment'] = df.apply(assign_segment, axis=1)

print(df['rfm_segment'].value_counts())

rfm_segment
Champions    20687
Lost         16102
Loyal        15898
At Risk      11313
Name: count, dtype: int64


In [37]:
rfm_summary = df.groupby(['rfm_segment', 'segment'])[['visit','conversion','spend']].mean().round(4)
print(rfm_summary)

                            visit  conversion   spend
rfm_segment segment                                  
At Risk     Mens E-Mail    0.1766      0.0140  1.5076
            No E-Mail      0.1031      0.0035  0.5247
            Womens E-Mail  0.1528      0.0082  1.0725
Champions   Mens E-Mail    0.2314      0.0180  2.0058
            No E-Mail      0.1422      0.0094  0.9567
            Womens E-Mail  0.1834      0.0116  1.2298
Lost        Mens E-Mail    0.1329      0.0080  0.9350
            No E-Mail      0.0647      0.0030  0.4574
            Womens E-Mail  0.1126      0.0060  0.6715
Loyal       Mens E-Mail    0.1751      0.0091  1.1050
            No E-Mail      0.1038      0.0054  0.5500
            Womens E-Mail  0.1474      0.0086  1.2924


## insights
### 1. sending men email to at-risk customer has highest uplift, saves customer(0.35% to 1.40% for conversion)
### 2. sending men email to lost-customer, also response good, so worth targeting(0.30% to 0.80%)
### 3. chapions don't need agggressive msg, they will convert anyway

# 3. Uplift Modeling

In [38]:
from sklearn.preprocessing import LabelEncoder

treatment = df[df['segment'] == df['segment'].map(lambda x:x)].copy()

df_uplift = df.copy()

for col in ['zip_code', 'channel', 'rfm_segment']:
    le = LabelEncoder()
    df_uplift[col] = le.fit_transform(df_uplift[col])

In [39]:
df_uplift = df_uplift[df_uplift['segment'].isin(['Mens E-Mail', 'No E-Mail'])]
df_uplift['treatment'] = (df_uplift['segment'] == 'Mens E-Mail').astype(int)

In [40]:
print(df_uplift.shape)
df_uplift['treatment'].value_counts()

(42613, 16)


treatment
1    21307
0    21306
Name: count, dtype: int64

In [41]:
from xgboost import XGBClassifier

features_uplift = ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code', 'channel', 'rfm_segment']

treated = df_uplift[df_uplift['treatment'] == 1]
control = df_uplift[df_uplift['treatment'] == 0]

model_treated = XGBClassifier(random_state=42, eval_metric='logloss')
model_treated.fit(treated[features_uplift], treated['conversion'])

model_control = XGBClassifier(random_state=42, eval_metric='logloss')
model_control.fit(control[features_uplift], control['conversion'])

print(f"Treatment model {len(treated)} ")
print(f"Control model {len(control)} ")

Treatment model 21307 
Control model 21306 


In [42]:
df_uplift['prob_treated'] = model_treated.predict_proba(df_uplift[features_uplift])[:, 1]

df_uplift['prob_control'] = model_control.predict_proba(df_uplift[features_uplift])[:, 1]

df_uplift['uplift_score'] = df_uplift['prob_treated'] - df_uplift['prob_control']

In [43]:
df_uplift[['recency','history','rfm_segment','prob_treated','prob_control','uplift_score']].head(10)

,recency,history,rfm_segment,prob_treated,prob_control,uplift_score
1,6,329.08,1,0.002284,0.014752,-0.012468
3,9,675.83,0,0.005572,0.000297,0.005275
8,9,675.07,0,0.018024,0.001053,0.016971
13,2,101.64,3,0.022318,0.009272,0.013047
14,4,241.42,1,0.001709,0.000608,0.001100
15,3,58.13,3,0.008694,0.031599,-0.022905
16,5,29.99,3,0.002010,0.001553,0.000457
17,9,112.35,2,0.000550,0.000643,-0.000094
18,11,219.04,0,0.005117,0.009895,-0.004778
19,5,828.42,1,0.067447,0.000589,0.066857


In [44]:
df_uplift['uplift_score'].describe()

count    42613.000000
mean         0.006711
std          0.032913
min         -0.649015
25%          0.000043
50%          0.002455
75%          0.008817
max          0.649573
Name: uplift_score, dtype: float64

In [45]:
# we are classifying customer into this segments for uplift segment:

# Uplift segment - what each means:
# Persuadable       : score > 0.02 - email genuinely moves them, Target these
# Sure Thing / Weak : score 0 - 0.02 - small or no effect, email not critical
# Sleeping Dog/Lost : score < 0 - email hurts conversion, Avoid email these

In [46]:
def uplift_segment(score):
    if score > 0.02:
        return 'Persuadable'
    elif score > 0:
        return 'Sure Thing / Weak'
    else:
        return 'Sleeping Dog / Lost'

df_uplift['uplift_segment'] = df_uplift['uplift_score'].apply(uplift_segment)

print(df_uplift['uplift_segment'].value_counts())

uplift_segment
Sure Thing / Weak      26891
Sleeping Dog / Lost    10414
Persuadable             5308
Name: count, dtype: int64


In [47]:
df_uplift.groupby(['uplift_segment', 'rfm_segment'])[['conversion']].mean().round(4)

conversion
uplift_segment      rfm_segment            
Persuadable         0                0.0519
                    1                0.0458
                    2                0.0411
                    3                0.0477
Sleeping Dog / Lost 0                0.0100
                    1                0.0181
                    2                0.0082
                    3                0.0080
Sure Thing / Weak   0                0.0000
                    1                0.0005
                    2                0.0010
                    3                0.0011

In [48]:
rfm_map = {0: 'At Risk', 1: 'Champions', 2: 'Lost', 3: 'Loyal'}
df_uplift['rfm_name'] = df_uplift['rfm_segment'].map(rfm_map)

df_uplift.groupby(['uplift_segment', 'rfm_name'])[['conversion']].mean().round(4)

conversion
uplift_segment      rfm_name             
Persuadable         At Risk        0.0519
                    Champions      0.0458
                    Lost           0.0411
                    Loyal          0.0477
Sleeping Dog / Lost At Risk        0.0100
                    Champions      0.0181
                    Lost           0.0082
                    Loyal          0.0080
Sure Thing / Weak   At Risk        0.0000
                    Champions      0.0005
                    Lost           0.0010
                    Loyal          0.0011

## Insights??

#### 1. At Risk + Persuadable = your #1 target
#### 5.19% conversion rate
#### High value customers who respond to email
#
#
#### 2. Lost + Persuadable = surprising opportunity
#### 4.11% conversion
#### Cold customers who still respond to right email

# 4. Business Dashboard

In [49]:
print("A/B TESTING RESULTS")
print("=" * 50)

ab_summary = df.groupby('segment')[['visit','conversion','spend']].mean().round(4)
ab_summary['conversion_uplift'] = (ab_summary['conversion'] - ab_summary.loc['No E-Mail','conversion']).round(4)
ab_summary['spend_uplift'] = (ab_summary['spend'] - ab_summary.loc['No E-Mail','spend']).round(4)
print(ab_summary)

A/B TESTING RESULTS
                visit  conversion   spend  conversion_uplift  spend_uplift
segment                                                                   
Mens E-Mail    0.1828      0.0125  1.4226             0.0068        0.7698
No E-Mail      0.1062      0.0057  0.6528             0.0000        0.0000
Womens E-Mail  0.1514      0.0088  1.0772             0.0031        0.4244


In [50]:
print("FINAL TARGETING RECOMMENDATION")
print("=" * 50)

final = df_uplift.groupby(['uplift_segment', 'rfm_name']).agg(
    customer_count=('uplift_score', 'count'),
    avg_uplift=('uplift_score', 'mean'),
    conversion_rate=('conversion', 'mean')).round(4)

print(final)

FINAL TARGETING RECOMMENDATION
                               customer_count  avg_uplift  conversion_rate
uplift_segment      rfm_name                                              
Persuadable         At Risk              1041      0.0612           0.0519
                    Champions            2557      0.0564           0.0458
                    Lost                  851      0.0451           0.0411
                    Loyal                 859      0.0507           0.0477
Sleeping Dog / Lost At Risk              1204     -0.0103           0.0100
                    Champions            3639     -0.0215           0.0181
                    Lost                 1959     -0.0099           0.0082
                    Loyal                3612     -0.0082           0.0080
Sure Thing / Weak   At Risk              5286      0.0053           0.0000
                    Champions            7483      0.0061           0.0005
                    Lost                 7926      0.0043           0